In [ ]:
import sys
sys.path.append("../eng-quant/")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [91]:
from relaxers.quadratic_model_lin_approx import quadratic_model_lin_approx # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np

In [92]:
LIN_APPROX = 10

# Whether to do an inner or outer approximation
OUTER_APPROXIMATION = False

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

In [93]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                            f_name="model11_quad_reform.json")
val, time, x = solve_lp_return_x()
print(x)
quad, q_time = solve_qcp(f_name="model11_quad_reform.json")

print(condition_number_nes_basic())
print(f"Error: {val - quad}")

[159.9999999967452, 142.8571428449557, 139.99999999896417, 99.99999999983207, 42.85714284568819, 157.14285715227948, 120.0000000012687, 80.00000000013581, 159.99999999559265, 199.9999999976252, 2999.7237332747027, 2456.969099733491, 2345.4441750683145, 3808.179859416307, 4239.999999917693, 3520.173381771249, 1941.1711634744552, 2196.7442754846443, 3443.865105458666, 3600.00000000035]
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 7 8845HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Presolve a model with 0 rows, 20 columns and 0 nonzeros (Min)
Model fingerprint: 0xdd0b2740
Model has 10 linear objective coefficients
Model has 20 quadratic constraints
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  QMatrix range    [2e-02, 2e-02]
  QLMatrix range   [1e+00, 2e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [5e+01,

In [94]:
# In this case there are 10 quadratic variables
# Define a set of functions for each

# TODO: It should intelligently search for the quadratic variables, not just assume they're at the start

quads = [[val] for val in x[:10]]
iters = 5

for i in range(iters):
    print(f"Iteration {i}")
    points_functions = []
    

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
            refinement_points = len(qs)
            uniform = np.linspace(lower, upper, num - refinement_points)
            for q in qs:
                idx = np.searchsorted(uniform, q)
                uniform = np.insert(uniform, idx, q)
            return uniform
        return np_uniform_add_qs

    for q in quads:
        points_functions.append(np_uniform_add_q_factory(q))

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + 1, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name="model11_quad_reform.json")

    # quadratic_lin_approx_no_surrogate_refine(1, OUTER_APPROXIMATION, points_function=points_functions,
    #                                             remove_division=REMOVE_DIVISION,
    #                                             f_name="model11_quad_reform_nofix.json")
    val, time, x = solve_lp_return_x()
    # print(f"x: {x}")
    # print(f"val: {val}")
    print(f"Error: {val - quad}")

    for j, qs in enumerate(quads):
        qs.append(x[j])

print(x)
print(f"Error: {val - quad}")

Iteration 0
Error: 8.588418390208062
Iteration 1
Error: 1.0672920541728672
Iteration 2
Error: 0.1907551402473473
Iteration 3
Error: 0.07699637256973801
Iteration 4
Error: -1.9272066310804803e-05
[160.00000035986528, 146.4101620792961, 139.99999999991863, 99.99999999999744, 46.410162099985385, 153.5898378412713, 120.00000000004977, 80.0000078321336, 160.00000035986528, 200.00000000376713, 2868.535957078942, 2361.4302950836527, 2310.4758433129073, 3751.403563775274, 4240.0, 3332.2092735794845, 1856.2773652900519, 2124.8622762544173, 3410.7456120283414, 3599.999999989303]
Error: -1.9272066310804803e-05
